
# TMaze Combined Pipeline

This notebook combines the TMaze DLC coordinate processing, neural length loading,
trajectory cleaning, movement detection, and trial segmentation into one place.

You can either:
- Run the functions from here directly, or
- Export the main code cell as a `.py` script for batch processing.


In [ ]:

import argparse
import glob
import json
import os
import re
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import h5py
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.signal import medfilt

# =========================
# Global parameters
# =========================

FPS = 25.0

# Cleaning / movement parameters
LIKELIHOOD_THRESHOLD = 0.9
MAX_JUMP = 5.0           # in same units as coordinates (px or cm)
SMOOTHING_SIGMA = 1.0
SPEED_THRESHOLD = 2.0    # pixels / frame threshold for movement (matches Linear-track pipeline)
MIN_SPEED_DURATION = 0.4 # seconds
MIN_TRIAL_DURATION_S = 0.5
MIN_DWELL_FRAMES = 4
MIN_DIR_COS = -0.5

# Port ROIs, by camera/apparatus era. Unlike the linear track, the T-maze apparatus is physically
# swapped in and out of the room, so its ports moved between all three eras (see CLAUDE.md
# "Reward outcome" section and docs/reward_extraction_plan.md) -- aging cohort (2024), young
# cohort pre-May-2026, and Mouse944/Mouse945-Round2 post-May-2026.
#
# Calibrated directly in DLC coordinate space (see the "Manually locate reward port ROIs --
# DLC-coordinate view" section below), not against raw video pixels: inside_port()/label_ports()
# match DLC-tracked (x, y), and DLC's coordinate space differs from the video frame's by a
# consistent ~400px vertical + ~20-45px horizontal offset (see that section for the fitted
# per-era offsets).
PORT_ROIS_BY_ERA = {
    "aging2024":   {"left": (90, 190, 50, 150), "right": (1590, 1690, 90, 190), "base": (831, 936, 870, 970)},
    "pre2026-05":  {"left": (90, 190, 10, 110), "right": (1590, 1690, 70, 170), "base": (831, 936, 820, 920)},
    "post2026-05": {"left": (75, 175, 10, 110), "right": (1590, 1690, 30, 130), "base": (845, 945, 810, 910)},
}


def era_for_session(session_dir) -> str:
    """Camera/apparatus era for a T-MAZE session -- three eras, unlike the linear track's two,
    because the T-maze apparatus itself moves between rounds. Matches
    detect_reward_flushes.py's tmaze_era_for_session() exactly.
    """
    m = re.search(r"(\d{4}-\d{2}-\d{2})T\d{2}_\d{2}_\d{2}", str(session_dir))
    if not m:
        raise ValueError(f"no timestamp found in session_dir: {session_dir}")
    date = m.group(1)
    if date < "2025-01-01":
        return "aging2024"
    return "post2026-05" if date >= "2026-05-01" else "pre2026-05"


# Backward-compatible default, used only by preview/plotting cells below that fall back to it
# when no explicit port_rois is passed -- the batch loop further down looks up the correct era
# per session instead of relying on this constant.
PORT_ROIS_DEFAULT = PORT_ROIS_BY_ERA["pre2026-05"]


def extract_datetime_stamp(path: str) -> str:
    """
    Extract 'YYYY-MM-DD_HH-MM-SS' from a path that contains
    'YYYY-MM-DDTHH_MM_SS' (e.g. 2024-10-06T13_17_01).
    """
    m = re.search(r"(\d{4}-\d{2}-\d{2})T(\d{2}_\d{2}_\d{2})", path)
    if m:
        rec_date = m.group(1)              # '2024-10-06'
        rec_time_raw = m.group(2)          # '13_17_01'
        rec_time = rec_time_raw.replace("_", "-")  # '13-17-01'
        return f"{rec_date}_{rec_time}"
    # fallback: just date if present
    m2 = re.search(r"(\d{4}-\d{2}-\d{2})", path)
    if m2:
        return m2.group(1)
    return "unknown_datetime"


def unit(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    return v / (n + 1e-9)


# =========================
# DLC helpers
# =========================

def pick_most_robust_bodypart(dlc_df: pd.DataFrame, thr: float = 0.9) -> str:
    """Return bodypart name with highest fraction of frames having likelihood > thr."""
    lcols = [
        c for c in dlc_df.columns
        if len(c) == 3 and str(c[2]).strip().lower() == "likelihood"
    ]
    if not lcols:
        raise ValueError("No likelihood columns found in DLC data.")
    scores = {}
    for col in lcols:
        bp = col[1]
        s = pd.to_numeric(dlc_df[col], errors="coerce")
        total = s.notna().sum()
        if total == 0:
            continue
        score = (s > thr).sum() / total
        scores[bp] = max(scores.get(bp, 0.0), float(score))
    if not scores:
        raise ValueError("No valid likelihood data to compute robustness.")
    return max(scores, key=scores.get)


def col_of(dlc_df: pd.DataFrame, bodypart: str, coord: str):
    """Return MultiIndex column for given bodypart and coord in {'x','y','likelihood'}."""
    cs = [
        c for c in dlc_df.columns
        if len(c) == 3 and c[1] == bodypart and str(c[2]).strip().lower() == coord
    ]
    if not cs:
        raise ValueError(f"Missing column for bodypart={bodypart}, coord={coord}.")
    return cs[0]


def load_trim_info(trim_csv_path: str) -> pd.Series:
    """
    Load dlc_trim_info.csv and return its first row.

    Expected columns:
      - 'start_frame'  (required)
      - optionally something like 'bodypart', 'best_bodypart', or 'robust_bodypart'
    """
    trim_df = pd.read_csv(trim_csv_path)
    if trim_df.empty:
        raise ValueError(f"{trim_csv_path} is empty.")
    if "start_frame" not in trim_df.columns:
        raise ValueError("dlc_trim_info.csv must contain a 'start_frame' column.")
    return trim_df.iloc[0]


def choose_bodypart_from_trim_info(
    trim_row: pd.Series,
    dlc_df: pd.DataFrame,
    fallback_thr: float = 0.9,
) -> str:
    """
    Try to read bodypart name from trim_row; if missing, fall back to
    pick_most_robust_bodypart on dlc_df.
    """
    for key in ["bodypart", "best_bodypart", "robust_bodypart", "most_robust_bodypart"]:
        if key in trim_row.index and isinstance(trim_row[key], str) and trim_row[key]:
            return trim_row[key]
    # Fallback
    return pick_most_robust_bodypart(dlc_df, thr=fallback_thr)


def load_dlc_for_bodypart(
    dlc_csv_path: str,
    bodypart: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Load DLC CSV and return (x, y, likelihood, full_df) for a single bodypart.
    """
    dlc_df = pd.read_csv(dlc_csv_path, header=[0, 1, 2])

    x_col = col_of(dlc_df, bodypart, "x")
    y_col = col_of(dlc_df, bodypart, "y")
    l_col = col_of(dlc_df, bodypart, "likelihood")

    x = pd.to_numeric(dlc_df[x_col], errors="coerce").to_numpy()
    y = pd.to_numeric(dlc_df[y_col], errors="coerce").to_numpy()
    p = pd.to_numeric(dlc_df[l_col], errors="coerce").to_numpy()

    return x, y, p, dlc_df


# =========================
# Neural length helper
# =========================

def load_neural_length(caiman_path: str) -> int:
    """
    Load neural activity length (number of frames).

    Tries CaImAn's cnmf.load_CNMF first; if that fails, falls back to reading
    '/estimates/F_dff' from the HDF5 file.
    """
    # Try CaImAn CNMF loader
    try:
        from caiman.source_extraction.cnmf.cnmf import load_CNMF  # type: ignore
        cnmf_obj = load_CNMF(caiman_path)
        F_dff = cnmf_obj.estimates.F_dff[cnmf_obj.estimates.idx_components]
        return int(F_dff.shape[1])
    except Exception:
        pass

    # Fallback: raw HDF5 access
    with h5py.File(caiman_path, "r") as f:
        # Heuristic: look for something like /estimates/F_dff
        if "estimates" in f and "F_dff" in f["estimates"]:
            dset = f["estimates"]["F_dff"]
            # Shape (n_cells, n_frames)
            return int(dset.shape[1])
        raise RuntimeError(
            f"Could not determine neural length from {caiman_path}. "
            "Install CaImAn or adjust load_neural_length."
        )


# =========================
# Trajectory cleaning
# =========================

def process_location_2D(
    location: np.ndarray,
    likelihood: np.ndarray,
    likelihood_threshold: float = LIKELIHOOD_THRESHOLD,
    max_jump: float = MAX_JUMP,
    smoothing_sigma: float = SMOOTHING_SIGMA,
) -> np.ndarray:
    """
    Basic trajectory cleaning:

    1) Remove samples with low likelihood.
    2) Remove large frame-to-frame jumps (> max_jump) by marking them as bad.
    3) Linearly interpolate over bad samples.
    4) Gaussian smooth each coordinate.

    Parameters
    ----------
    location   : (N, 2) array of [x, y]
    likelihood : (N,) array of DLC likelihoods for that bodypart

    Returns
    -------
    cleaned : (N, 2) array
    """
    loc = np.asarray(location, float)
    lh = np.asarray(likelihood, float).squeeze()

    if loc.ndim != 2 or loc.shape[1] != 2:
        raise ValueError("location must be (N, 2) array")
    if loc.shape[0] != lh.shape[0]:
        raise ValueError("location and likelihood must have same length")

    n = loc.shape[0]

    # 1) Mark low-likelihood samples as bad
    bad = ~(lh >= likelihood_threshold)

    # 2) Mark large jumps as bad
    dx = np.diff(loc[:, 0])
    dy = np.diff(loc[:, 1])
    jump = np.hypot(dx, dy)
    jump_bad = np.concatenate([[False], jump > max_jump])
    bad |= jump_bad

    # 3) Interpolate over bad samples separately for x and y
    cleaned = loc.copy()
    for dim in range(2):
        arr = cleaned[:, dim]
        good_idx = np.where(~bad & ~np.isnan(arr))[0]
        bad_idx = np.where(bad | np.isnan(arr))[0]

        if len(good_idx) == 0:
            # fall back: nothing we can do
            continue

        # clamp endpoints
        if len(bad_idx) > 0:
            if bad_idx[0] == 0:
                first_good = good_idx[0]
                arr[0:first_good] = arr[first_good]
            if bad_idx[-1] == n - 1:
                last_good = good_idx[-1]
                arr[last_good:] = arr[last_good]

        bad_idx = np.where(np.isnan(arr))[0]
        if len(bad_idx) > 0:
            arr[bad_idx] = np.interp(bad_idx, good_idx, arr[good_idx])

        cleaned[:, dim] = arr

    # 4) Smooth
    for dim in range(2):
        cleaned[:, dim] = gaussian_filter1d(cleaned[:, dim], sigma=smoothing_sigma)

    return cleaned


def get_movement_mask_2D(
    location: np.ndarray,
    speed_threshold: float = SPEED_THRESHOLD,
    min_speed_duration: float = MIN_SPEED_DURATION,
    samples_per_second: float = FPS,
) -> Tuple[np.ndarray, List[Tuple[int, int]]]:
    """
    Compute a boolean mask of "movement" frames from 2D coordinates.

    Parameters
    ----------
    location          : (N, 2) array of [x, y]
    speed_threshold   : speed (pixels/frame) above which a frame counts as "moving"
    min_speed_duration: minimum contiguous duration (in s) to keep as movement
    samples_per_second: sampling rate

    Returns
    -------
    movement_mask     : (N,) boolean array
    movement_periods  : list of (start_idx, end_idx) (inclusive)
    """
    loc = np.asarray(location, float)
    if loc.ndim != 2 or loc.shape[1] != 2:
        raise ValueError("location must be (N, 2) array")

    vx = np.gradient(loc[:, 0])
    vy = np.gradient(loc[:, 1])
    speed = np.hypot(vx, vy)  # pixels / frame, matches the Linear-track pipeline

    # Boolean: speed above threshold, per frame
    speed_bool = speed >= speed_threshold

    # Find contiguous segments of True
    periods: List[Tuple[int, int]] = []
    in_run = False
    start = 0
    for i, val in enumerate(speed_bool):
        if val and not in_run:
            in_run = True
            start = i
        elif not val and in_run:
            in_run = False
            periods.append((start, i - 1))
    if in_run:
        periods.append((start, len(speed_bool) - 1))

    # Keep only sufficiently long periods
    min_samples = int(np.round(min_speed_duration * samples_per_second))
    filtered_periods = [
        (a, b) for (a, b) in periods if (b - a + 1) >= min_samples
    ]

    # Build mask
    mask = np.zeros(loc.shape[0], dtype=bool)
    for a, b in filtered_periods:
        mask[a:b + 1] = True

    return mask, filtered_periods


def apply_bounding_boxes(
    location: np.ndarray,
    boxes: Optional[Sequence[Tuple[float, float, float, float]]] = None,
) -> np.ndarray:
    """
    Optionally zero-out points outside user-defined bounding boxes.

    boxes is a sequence of (xmin, xmax, ymin, ymax). A point is kept if it is inside
    *any* box. If boxes is None or empty, the input is returned unchanged.
    """
    loc = np.asarray(location, float)
    if boxes is None or len(list(boxes)) == 0:
        return loc

    mask_any = np.zeros(loc.shape[0], dtype=bool)
    for xmin, xmax, ymin, ymax in boxes:
        inside = (
            (loc[:, 0] >= xmin) & (loc[:, 0] <= xmax) &
            (loc[:, 1] >= ymin) & (loc[:, 1] <= ymax)
        )
        mask_any |= inside

    loc_filtered = loc.copy()
    loc_filtered[~mask_any, :] = np.nan
    return loc_filtered


# =========================
# Port labeling and trials
# =========================

def roi_center(roi: Tuple[float, float, float, float]) -> Tuple[float, float]:
    """Center point of a (xmin, xmax, ymin, ymax) rectangular ROI."""
    xmin, xmax, ymin, ymax = roi
    return (xmin + xmax) / 2.0, (ymin + ymax) / 2.0


def inside_port(xy: Sequence[float], name: str, port_rois: Dict[str, Tuple[float, float, float, float]]) -> bool:
    xmin, xmax, ymin, ymax = port_rois[name]
    return xmin <= xy[0] <= xmax and ymin <= xy[1] <= ymax


def label_ports(
    XY: np.ndarray,
    valid_mask: np.ndarray,
    port_rois: Dict[str, Tuple[float, float, float, float]],
    min_dwell_frames: int = MIN_DWELL_FRAMES,
) -> np.ndarray:
    """
    Return array of port labels ('base', 'left', 'right', or None) for each frame.
    """
    labels = np.empty(len(XY), dtype=object)
    current = None
    dwell = 0

    for i, xy in enumerate(XY):
        if not valid_mask[i] or np.any(np.isnan(xy)):
            labels[i] = current
            continue

        hit = None
        for nm in ("base", "left", "right"):
            if nm in port_rois and inside_port(xy, nm, port_rois):
                hit = nm
                break

        if hit == current:
            if hit is not None:
                dwell += 1
        else:
            if hit is None:
                current = None
                dwell = 0
            else:
                dwell += 1
                if dwell >= min_dwell_frames:
                    current = hit
        labels[i] = current

    return labels


def classify_trial(origin: str, dest: str) -> str:
    if origin == "base" and dest == "left":
        return "base_to_left"
    if origin == "left" and dest == "base":
        return "left_to_base"
    if origin == "base" and dest == "right":
        return "base_to_right"
    if origin == "right" and dest == "base":
        return "right_to_base"
    if origin == "left" and dest == "right":
        return "left_to_right"
    if origin == "right" and dest == "left":
        return "right_to_left"
    return "other"


def build_dir_templates(port_rois: Dict[str, Tuple[float, float, float, float]]) -> Dict[Tuple[str, str], np.ndarray]:
    pairs = [
        ("base", "left"),
        ("left", "base"),
        ("base", "right"),
        ("right", "base"),
        ("left", "right"),
        ("right", "left"),
    ]
    templates: Dict[Tuple[str, str], np.ndarray] = {}
    for origin, dest in pairs:
        if origin not in port_rois or dest not in port_rois:
            continue
        ox, oy = roi_center(port_rois[origin])
        dx, dy = roi_center(port_rois[dest])
        templates[(origin, dest)] = unit(np.array([dx - ox, dy - oy]))
    return templates


def segment_trials_from_trajectory(
    XY: np.ndarray,
    likelihood: np.ndarray,
    port_rois: Dict[str, Tuple[float, float, float, float]],
    fps: float = FPS,
    min_trial_duration_s: float = MIN_TRIAL_DURATION_S,
    min_dir_cos: float = MIN_DIR_COS,
    speed_min: float = 0.0,
) -> pd.DataFrame:
    """
    Trial segmentation logic adapted to work on an already-trimmed trajectory.

    Returns a DataFrame with (start_frame, end_frame, origin, destination,
    duration_s, mean_speed, dir_score, trial_type) in *local* frame indices.
    """
    # Compute speed
    x_s = XY[:, 0]
    y_s = XY[:, 1]

    # Basic smoothing for speed only
    x_med = medfilt(x_s, kernel_size=5)
    y_med = medfilt(y_s, kernel_size=5)

    vx = np.gradient(x_med) * fps
    vy = np.gradient(y_med) * fps
    speed_raw = np.hypot(vx, vy)

    # Likelihood / movement masks
    good_lik = likelihood >= LIKELIHOOD_THRESHOLD
    moving_raw = speed_raw >= speed_min
    valid = good_lik & moving_raw

    # Port labels
    port_label = label_ports(XY, good_lik, port_rois)
    dir_templates = build_dir_templates(port_rois)
    min_trial_len = int(min_trial_duration_s * fps)

    trials: List[Dict[str, object]] = []
    state_port = port_label[0]
    in_transit = False
    origin = None
    start_idx: Optional[int] = None

    for i in range(1, len(port_label)):
        cur = port_label[i]
        if not in_transit:
            if state_port is not None and cur is None:
                in_transit = True
                origin = state_port
                start_idx = i - 1
        else:
            if cur is not None:
                dest = cur
                end_idx = i

                if (
                    origin is not None
                    and dest is not None
                    and origin != dest
                    and (origin, dest) in dir_templates
                ):
                    seg = slice(start_idx, end_idx + 1)
                    seg_valid = valid[seg] & (speed_raw[seg] > speed_min)

                    if seg_valid.sum() >= 3 and (end_idx - start_idx + 1) >= min_trial_len:
                        vx_seg = vx[seg][seg_valid]
                        vy_seg = vy[seg][seg_valid]
                        sp_seg = speed_raw[seg][seg_valid]

                        templ = dir_templates[(origin, dest)]
                        vvec = np.column_stack([vx_seg, vy_seg])
                        cos_vals = (vvec @ templ) / (
                            np.linalg.norm(vvec, axis=1) + 1e-9
                        )
                        mean_cos = float(np.nanmean(cos_vals))

                        if mean_cos >= min_dir_cos:
                            trials.append(
                                {
                                    "start_frame": int(start_idx),
                                    "end_frame": int(end_idx),
                                    "origin": origin,
                                    "destination": dest,
                                    "duration_s": (end_idx - start_idx + 1) / fps,
                                    "mean_speed": float(np.nanmean(sp_seg)),
                                    "dir_score": mean_cos,
                                    "trial_type": classify_trial(origin, dest),
                                }
                            )

                in_transit = False
                origin = None
                start_idx = None

        state_port = cur

    # Always return the expected columns, even with zero trials -- an empty
    # pd.DataFrame(trials) would otherwise have no columns at all, breaking every
    # downstream trials_df['start_frame'] access for sessions with no valid trials.
    return pd.DataFrame(trials, columns=["start_frame", "end_frame", "origin", "destination", "duration_s", "mean_speed", "dir_score", "trial_type"])


# =========================
# Session-level pipeline
# =========================

def find_caiman_results(session_dir: str) -> str:
    """
    Look for caiman_results.hdf5 under the standard caiman output folders.

    TMaze sessions are aging-cohort only. caiman_aging/ is the current
    best-parameter reprocessing and is preferred whenever present; caiman_final/
    is kept as a fallback for any session not yet reprocessed with aging-specific
    gSig/gSig_filt.
    """
    for subdir in ("caiman_aging", "caiman_final"):
        guess = os.path.join(session_dir, subdir, "caiman_results.hdf5")
        if os.path.isfile(guess):
            return guess
    raise FileNotFoundError(
        f"caiman_results.hdf5 not found under {session_dir}/caiman_aging/ or "
        f"{session_dir}/caiman_final/; please pass caiman_path explicitly."
    )


def process_session(
    session_dir: str,
    trim_info_name: str = "dlc_trim_info.csv",
    dlc_pattern: str = "*_1100000_filtered.csv",
    caiman_path: Optional[str] = None,
    port_rois: Optional[Dict[str, Tuple[float, float, float, float]]] = None,
    bounding_boxes: Optional[Sequence[Tuple[float, float, float, float]]] = None,
    output_dir: Optional[str] = None,
) -> None:
    """
    Run the full pipeline for a single session.

    Parameters
    ----------
    session_dir   : directory containing 'dlc_trim_info.csv' and a 'dlc' subfolder
    trim_info_name: filename of dlc_trim_info.csv
    dlc_pattern   : glob pattern to find the DLC CSV inside 'dlc' subfolder
    caiman_path   : path to caiman_results.hdf5 (if None, guess inside session_dir)
    port_rois     : dict like {'base': (x,y,r), ...}; if None, PORT_ROIS_DEFAULT is used
    bounding_boxes: optional list of (xmin, xmax, ymin, ymax) to keep;
                    points outside all boxes are set to NaN *before* segmentation
    output_dir    : where to write outputs; if None, use 'dlc' subfolder of session_dir
    """
    session_dir = os.path.abspath(session_dir)

    # 1) Paths
    trim_info_path = os.path.join(session_dir, trim_info_name)
    dlc_dir = os.path.join(session_dir, "dlc")
    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError(f"No DLC files matching {dlc_pattern!r} in {dlc_dir}")
    if len(dlc_files) > 1:
        print("Warning: multiple DLC files found, using the first one.")
    dlc_csv_path = dlc_files[0]

    if caiman_path is None:
        caiman_path = find_caiman_results(session_dir)

    if output_dir is None:
        output_dir = dlc_dir
    os.makedirs(output_dir, exist_ok=True)

    # 2) Neural length
    neural_len = load_neural_length(caiman_path)
    print(f"[{session_dir}] neural_length = {neural_len} frames")

    # 3) DLC + trim info
    trim_row = load_trim_info(trim_info_path)
    # Align to where the clip ACTUALLY starts, not where the trim was asked to start.
    # trim_videos_from_dlc.py cuts with `ffmpeg -ss <t> -i raw ... -c copy`, and input
    # seeking with stream copy cannot begin mid-GOP: ffmpeg backs off to an earlier
    # keyframe, so the clip starts k frames BEFORE `start_frame` (k measured per session
    # by scripts/data_audit/measure_trim_offset.py; 0-57 frames observed). Slicing the DLC
    # table at `start_frame` pairs pose row i with a neural frame k frames earlier, i.e.
    # behaviour runs AHEAD of the neural data by up to ~2 s, the same direction every
    # session. Fall back to `start_frame` for sessions not yet measured.
    if "start_frame_actual" in trim_row.index and pd.notna(trim_row["start_frame_actual"]):
        trim_start_frame = int(trim_row["start_frame_actual"])
    else:
        trim_start_frame = int(trim_row["start_frame"])
    print(f"[{session_dir}] trim_start_frame = {trim_start_frame}")

    # load DLC once to resolve bodypart and coordinates
    dlc_df_full = pd.read_csv(dlc_csv_path, header=[0, 1, 2])
    bodypart = choose_bodypart_from_trim_info(trim_row, dlc_df_full, fallback_thr=LIKELIHOOD_THRESHOLD)
    print(f"[{session_dir}] using bodypart = {bodypart!r}")

    x_full = pd.to_numeric(dlc_df_full[col_of(dlc_df_full, bodypart, "x")], errors="coerce").to_numpy()
    y_full = pd.to_numeric(dlc_df_full[col_of(dlc_df_full, bodypart, "y")], errors="coerce").to_numpy()
    p_full = pd.to_numeric(dlc_df_full[col_of(dlc_df_full, bodypart, "likelihood")], errors="coerce").to_numpy()

    n_frames_full = len(x_full)
    trim_end_frame = min(trim_start_frame + neural_len, n_frames_full)
    if trim_end_frame <= trim_start_frame:
        raise ValueError("Trim window is empty; check start_frame and neural length.")

    # 4) Trim to recording window
    sl = slice(trim_start_frame, trim_end_frame)
    x_trim = x_full[sl]
    y_trim = y_full[sl]
    p_trim = p_full[sl]

    n_trim = len(x_trim)
    print(f"[{session_dir}] trimmed DLC frames: {n_trim} (global {trim_start_frame}..{trim_end_frame-1})")

    # global frame indices for trimmed window
    global_frames = np.arange(trim_start_frame, trim_end_frame, dtype=int)
    local_frames = np.arange(n_trim, dtype=int)

    # Raw location (as 2D coordinates)
    loc_raw = np.column_stack([x_trim, y_trim])

    # 5) Clean trajectory
    loc_clean = process_location_2D(
        loc_raw,
        p_trim,
        likelihood_threshold=LIKELIHOOD_THRESHOLD,
        max_jump=MAX_JUMP,
        smoothing_sigma=SMOOTHING_SIGMA,
    )

    # Optional bounding-box filtering
    loc_clean = apply_bounding_boxes(loc_clean, bounding_boxes)

    # Movement mask
    movement_mask, movement_periods = get_movement_mask_2D(
        loc_clean,
        speed_threshold=SPEED_THRESHOLD,
        min_speed_duration=MIN_SPEED_DURATION,
        samples_per_second=FPS,
    )

    # 6) Port labeling and trial segmentation
    if port_rois is None:
        port_rois = PORT_ROIS_DEFAULT.copy()

    port_labels = label_ports(loc_clean, valid_mask=(p_trim >= LIKELIHOOD_THRESHOLD), port_rois=port_rois)
    trials_df_local = segment_trials_from_trajectory(
        loc_clean,
        p_trim,
        port_rois=port_rois,
        fps=FPS,
        min_trial_duration_s=MIN_TRIAL_DURATION_S,
        min_dir_cos=MIN_DIR_COS,
        speed_min=0.0,
    )

    # Add global frame indices to trials
    trials_df = trials_df_local.copy()
    trials_df["start_frame_global"] = trials_df["start_frame"] + trim_start_frame
    trials_df["end_frame_global"] = trials_df["end_frame"] + trim_start_frame

    # 7) Save outputs
    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_TMaze"

    # 7a) Frame-wise raw & cleaned positions for trimmed recording
    df_frames = pd.DataFrame(
        {
            "frame_local": local_frames,
            "frame_global": global_frames,
            "x_raw": x_trim,
            "y_raw": y_trim,
            "likelihood": p_trim,
            "x_clean": loc_clean[:, 0],
            "y_clean": loc_clean[:, 1],
            "moving": movement_mask.astype(int),
            "port_label": port_labels,
        }
    )
    frames_path = os.path.join(output_dir, f"{prefix}_trimmed_positions_and_ports.csv")
    df_frames.to_csv(frames_path, index=False)

    # 7b) Trials table (local and global indices)
    trials_path = os.path.join(output_dir, f"{prefix}_trimmed_trials.csv")
    trials_df.to_csv(trials_path, index=False)

    # 7c) Trial-by-trial raw locations (long format)
    rows_raw = []
    rows_clean = []
    for trial_id, tr in trials_df.iterrows():
        s_loc = int(tr["start_frame"])
        e_loc = int(tr["end_frame"])
        s_glob = int(tr["start_frame_global"])
        e_glob = int(tr["end_frame_global"])

        # local indices inside trimmed window
        fr_local = np.arange(s_loc, e_loc + 1, dtype=int)
        fr_global = np.arange(s_glob, e_glob + 1, dtype=int)
        fr_trial = np.arange(len(fr_local), dtype=int)

        # raw
        for fl, fg, ft, x, y in zip(
            fr_local,
            fr_global,
            fr_trial,
            x_trim[s_loc:e_loc + 1],
            y_trim[s_loc:e_loc + 1],
        ):
            rows_raw.append(
                {
                    "trial_id": int(trial_id),
                    "trial_type": tr["trial_type"],
                    "origin": tr["origin"],
                    "destination": tr["destination"],
                    "frame_in_trial": int(ft),
                    "frame_local": int(fl),
                    "frame_global": int(fg),
                    "x_raw": float(x),
                    "y_raw": float(y),
                }
            )

        # cleaned
        for fl, fg, ft, x, y in zip(
            fr_local,
            fr_global,
            fr_trial,
            loc_clean[s_loc:e_loc + 1, 0],
            loc_clean[s_loc:e_loc + 1, 1],
        ):
            rows_clean.append(
                {
                    "trial_id": int(trial_id),
                    "trial_type": tr["trial_type"],
                    "origin": tr["origin"],
                    "destination": tr["destination"],
                    "frame_in_trial": int(ft),
                    "frame_local": int(fl),
                    "frame_global": int(fg),
                    "x_clean": float(x),
                    "y_clean": float(y),
                }
            )

    df_trials_raw = pd.DataFrame(rows_raw)
    df_trials_clean = pd.DataFrame(rows_clean)

    trials_raw_path = os.path.join(output_dir, f"{prefix}_trimmed_trial_locations_raw.csv")
    trials_clean_path = os.path.join(output_dir, f"{prefix}_trimmed_trial_locations_clean.csv")

    df_trials_raw.to_csv(trials_raw_path, index=False)
    df_trials_clean.to_csv(trials_clean_path, index=False)

    print("Saved:")
    print("  ", frames_path)
    print("  ", trials_path)
    print("  ", trials_raw_path)
    print("  ", trials_clean_path)


print("Notebook functions loaded. Use process_session(session_dir=...) to run the pipeline.")


## Manually locate reward port ROIs (all three eras)

`PORT_ROIS_BY_ERA`'s current values put every port center outside the
physically tracked area for all three eras (verified empirically: 0.00%
of tracked frames fall inside any of the 9 circles across aging2024,
pre2026-05, and post2026-05 -- see the coverage check in the cell below).
This section overlays the *current* guess directly on a real video frame
for one representative session per era, so the circles can be dragged into
place by eye against the actual apparatus instead of against DLC scatter.

**Workflow:** run the cell below, look at where each dashed circle sits
relative to the real port in the image, edit the corresponding `(x, y, r)`
in `CANDIDATE_ROIS` right in this cell, and re-run -- repeat until each
circle sits on its port. Once all three eras look right, copy
`CANDIDATE_ROIS` up into `PORT_ROIS_BY_ERA` in the cell above.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# One representative session per era -- pick any session from that era's
# mice if these particular sessions ever move/get renamed.
ERA_EXAMPLES = {
    "aging2024": {
        "video": "/media/toor/SeagateClean/Mouse163/Round1.5/TMaze/2024-06-11T13_08_57/TMaze_behavior2024-06-11T13_08_57.avi",
        "frame_idx": 500,
    },
    "pre2026-05": {
        "video": "/media/toor/Seagate3/YOUNG_MICE/Mouse946/Round1.5/TMaze/2026-02-10T15_59_47/TMaze_Linear2026-02-10T15_59_47.avi",
        "frame_idx": 500,
    },
    "post2026-05": {
        "video": "/media/toor/Seagate3/Mouse945 - Round2/TMaze/2026-07-16T14_32_41/TMaze_Linear2026-07-16T14_32_41.avi",
        "frame_idx": 500,
    },
}

# Edit these and re-run the cell to move the circles -- start from whatever
# PORT_ROIS_BY_ERA currently has, or type in fresh numbers by eye.
CANDIDATE_ROIS = {
    "aging2024":   dict(PORT_ROIS_BY_ERA["aging2024"]),
    "pre2026-05":  dict(PORT_ROIS_BY_ERA["pre2026-05"]),
    "post2026-05": dict(PORT_ROIS_BY_ERA["post2026-05"]),
}

ROI_COLORS = {"base": "#C97524", "left": "#1B6FA8", "right": "#3F9142"}


def grab_frame(video_path: str, frame_idx: int):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idx = min(frame_idx, max(n_frames - 1, 0))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame_bgr = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"Could not read frame {idx} from {video_path}")
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, era in zip(axes, ["aging2024", "pre2026-05", "post2026-05"]):
    info = ERA_EXAMPLES[era]
    frame = grab_frame(info["video"], info["frame_idx"])
    ax.imshow(frame)
    for name, (xmin, xmax, ymin, ymax) in CANDIDATE_ROIS[era].items():
        rect = Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, linewidth=2.5, linestyle="--",
                          edgecolor=ROI_COLORS.get(name, "white"))
        ax.add_patch(rect)
        cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
        half_h = (ymax - ymin) / 2
        ax.annotate(f"{name}\n({xmin:.0f}-{xmax:.0f}, {ymin:.0f}-{ymax:.0f})", (cx, cy), color=ROI_COLORS.get(name, "white"),
                    ha="center", va="center", fontsize=9, fontweight="bold",
                    xytext=(0, -(half_h + 22)), textcoords="offset points")
    ax.set_title(f"{era}  --  {os.path.basename(info['video'])}", fontsize=10)
    ax.set_xlim(0, frame.shape[1])
    ax.set_ylim(frame.shape[0], 0)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Manually locate reward port ROIs -- DLC-coordinate view

**This is the view that actually matters for segmentation.** `inside_port()` /
`label_ports()` test the ROI rectangles against DLC-tracked (x, y), not against
raw video pixels -- and DLC's coordinate space is *not* the same as the video
frame's: across all three eras, no DLC bodypart's y ever exceeds ~1035, while the
video frame overlay above was calibrated against the full 1464px-tall raw frame.
That mismatch (a consistent ~400px vertical + ~20-45px horizontal offset,
dataset-wide) is exactly why the video-frame-calibrated boxes produce 0% coverage
/ zero trials at runtime.

**Workflow** (same as `Linear_combined_pipeline.ipynb`'s ROI-tuning section):
1. Set `DLC_ERA` in the first cell below to the era you're calibrating.
2. Edit `test_rois` in the last cell to move the boxes, re-run, and watch the
   printed coverage -- each port should land somewhere in the single-digit-to-
   teens percent range (`base` can run higher, since the animal returns there
   between every trial), not 0%.
3. Repeat per era, then send the final numbers over to save into `PORT_ROIS_BY_ERA`.

In [ ]:
import pandas as pd
import numpy as np

# 1. Path to your DLC CSV -- one option per camera/apparatus era. Switch DLC_ERA
# to check a different era's calibration.
DLC_PATH_OPTIONS = {
    "aging2024":   "/media/toor/SeagateClean/Mouse1636/Round1.5/TMaze/2024-06-11T14_57_14/dlc/behavior2024-06-11T14_57_14DLC_resnet50_T-MazeJun16shuffle1_1100000_filtered.csv",
    "pre2026-05":  "/media/toor/Seagate3/YOUNG_MICE/Mouse946/Round1.5/TMaze/2026-02-28T18_13_07/dlc/Linear2026-02-28T18_13_07DLC_resnet50_T-MazeJun16shuffle1_1100000_filtered.csv",
    "post2026-05": "/media/toor/Seagate3/Mouse945 - Round2/TMaze/2026-07-19T16_15_06/dlc/Linear2026-07-19T16_15_06DLC_resnet50_T-MazeJun16shuffle1_1100000_filtered.csv",
}
DLC_ERA = "post2026-05"  # <-- change to "pre2026-05" or "post2026-05" to check those instead
dlc_path = DLC_PATH_OPTIONS[DLC_ERA]

# 2. Load DLC CSV with 3-row MultiIndex header (scorer, bodypart, coord)
dlc_df = pd.read_csv(dlc_path, header=[0, 1, 2])
print("DLC shape:", dlc_df.shape)
print("Columns example:", dlc_df.columns[:5])

# 3. List available bodyparts, with mean likelihood (skips the non-bodypart
# "bodyparts" header column some DLC exports carry)
bodyparts = sorted({col[1] for col in dlc_df.columns if len(col) == 3 and col[1] != "bodyparts"})
mean_likelihood = {}
for bp in bodyparts:
    lik_col = [c for c in dlc_df.columns if len(c) == 3 and c[1] == bp and str(c[2]).lower() == "likelihood"][0]
    mean_likelihood[bp] = pd.to_numeric(dlc_df[lik_col], errors="coerce").mean()

print("\nAvailable bodyparts (mean likelihood):")
for bp in bodyparts:
    print(f"  - {bp}: {mean_likelihood[bp]:.3f}")

# 4. Automatically use the bodypart with the highest average likelihood
#    (override by setting bodypart explicitly, e.g. bodypart = "tail_base")
bodypart = max(mean_likelihood, key=mean_likelihood.get)
print(f"\nUsing bodypart: {bodypart!r} (mean likelihood {mean_likelihood[bodypart]:.3f})")

# 5. Helper to find the x/y columns for this bodypart
def col_of(dlc_df, bodypart, coord):
    coord = coord.lower()
    for col in dlc_df.columns:
        if len(col) == 3 and col[1] == bodypart and str(col[2]).lower() == coord:
            return col
    raise KeyError(f"Could not find coord={coord!r} for bodypart={bodypart!r}")

x_col = col_of(dlc_df, bodypart, "x")
y_col = col_of(dlc_df, bodypart, "y")
lik_col = col_of(dlc_df, bodypart, "likelihood")

x_raw = pd.to_numeric(dlc_df[x_col], errors="coerce").to_numpy()
y_raw = pd.to_numeric(dlc_df[y_col], errors="coerce").to_numpy()
lik_raw = pd.to_numeric(dlc_df[lik_col], errors="coerce").to_numpy()

XY_raw = np.column_stack([x_raw, y_raw])

print("\nXY_raw shape:", XY_raw.shape)
print("x range:", np.nanmin(x_raw), "\u2192", np.nanmax(x_raw))
print("y range:", np.nanmin(y_raw), "\u2192", np.nanmax(y_raw))
print("likelihood range:", np.nanmin(lik_raw), "\u2192", np.nanmax(lik_raw))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd

def plot_with_rois(XY, port_rois, subsample=5, title="ROI test", ylim=None, xlim=None):
    """
    XY        : (N,2) array of x,y raw or cleaned positions
    port_rois : dict{name: (xmin, xmax, ymin, ymax)}
    ylim, xlim: optional (lo, hi) in data coordinates, to zoom into a region
                instead of showing the whole frame.
    """
    XY = np.asarray(XY, float)
    fig, ax = plt.subplots(figsize=(7,7))

    # subsample to avoid heavy plotting
    idx = np.arange(0, len(XY), subsample)

    ax.scatter(XY[idx,0], XY[idx,1], s=2, alpha=0.3, label="trajectory")

    # draw each roi
    for name, (xmin, xmax, ymin, ymax) in port_rois.items():
        rect = Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, linewidth=2, color="red")
        ax.add_patch(rect)
        cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
        ax.text(cx, cy, name, color="red", ha="center", va="center", fontsize=10)

    ax.set_aspect("equal", adjustable="box")
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.set_ylim(*(ylim if ylim is not None else (XY[:,1].min(), XY[:,1].max())))
    ax.invert_yaxis()     # matches DLC coordinate system
    ax.set_title(title)
    plt.show()


def print_roi_coverage(XY, port_rois):
    """
    Prints how many frames fall within each ROI
    """
    XY = np.asarray(XY, float)
    n = len(XY)
    print(f"Total frames: {n}")

    for name, (xmin, xmax, ymin, ymax) in port_rois.items():
        inside = (XY[:,0] >= xmin) & (XY[:,0] <= xmax) & (XY[:,1] >= ymin) & (XY[:,1] <= ymax)
        print(f"{name:10s}: {inside.sum():6d} frames ({inside.mean()*100:5.2f} %)")

In [ ]:
# Starting guesses (you will adjust these) -- offset-corrected from the
# video-pixel-space PORT_ROIS_BY_ERA into DLC coordinate space (see the markdown
# note above for why the two spaces disagree by ~400px). These are for whichever
# era DLC_ERA is set to above; swap in the matching row if you change it:
#
#   aging2024:    {"left": (56, 161, 48, 153),   "right": (1631, 1736, 88, 193),  "base": (831, 936, 898, 1003)}
#   pre2026-05:   {"left": (76, 181, -14, 91),   "right": (1656, 1761, 36, 141),  "base": (851, 956, 836, 941)}
#   post2026-05:  {"left": (59, 164, -6, 99),    "right": (1659, 1764, 14, 119),  "base": (874, 979, 819, 924)}
test_rois = {
    "left":  (75, 175, 10, 110),     # (xmin, xmax, ymin, ymax)
    "right": (1590, 1690, 30, 130),
    "base":  (845, 945, 810, 910),
}

# Plot
plot_with_rois(XY_raw, test_rois, subsample=1, title="ROI Tuning Plot")

# Frame counts inside each ROI
print_roi_coverage(XY_raw, test_rois)

In [ ]:
# =========================
# Linearization helpers
# =========================

def _project_onto_segment(p, a, b):
    """
    Project point p onto line segment a-b.
    Returns (projection_point, t_clipped, squared_distance).
    t_clipped in [0,1] gives the relative position along the segment.
    """
    p = np.asarray(p, float)
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    ab = b - a
    ab2 = float(np.dot(ab, ab))
    if ab2 == 0.0:
        return a, 0.0, float(np.sum((p - a) ** 2))
    t = float(np.dot(p - a, ab) / ab2)
    t_clipped = float(np.clip(t, 0.0, 1.0))
    proj = a + t_clipped * ab
    dist2 = float(np.sum((p - proj) ** 2))
    return proj, t_clipped, dist2


def linearize_positions_along_polyline(location, poly_points):
    """
    Map 2D positions onto a 1D coordinate along a polyline.

    Parameters
    ----------
    location   : (N, 2) array of [x, y] positions
    poly_points: (M, 2) array of polyline vertices, M >= 2

    Returns
    -------
    linear_pos : (N,) array of distances along polyline (0 at first vertex),
                 NaN where input position is NaN.
    """
    loc = np.asarray(location, float)
    poly = np.asarray(poly_points, float)

    if loc.ndim != 2 or loc.shape[1] != 2:
        raise ValueError("location must be (N, 2)")
    if poly.ndim != 2 or poly.shape[1] != 2 or poly.shape[0] < 2:
        raise ValueError("poly_points must be (M, 2) with M >= 2")

    seg = np.diff(poly, axis=0)
    seg_len = np.linalg.norm(seg, axis=1)
    cum_len = np.concatenate([[0.0], np.cumsum(seg_len)])

    N = loc.shape[0]
    linear_pos = np.full(N, np.nan, float)

    for i in range(N):
        p = loc[i]
        if np.any(np.isnan(p)):
            continue

        best_dist2 = np.inf
        best_s = np.nan

        for s_idx in range(len(seg_len)):
            a = poly[s_idx]
            b = poly[s_idx + 1]
            _, t, d2 = _project_onto_segment(p, a, b)
            if d2 < best_dist2:
                best_dist2 = d2
                best_s = cum_len[s_idx] + t * seg_len[s_idx]

        linear_pos[i] = best_s

    return linear_pos


def _polyline_total_length(poly):
    poly = np.asarray(poly, float)
    seg = np.diff(poly, axis=0)
    return float(np.linalg.norm(seg, axis=1).sum())


In [ ]:
def _compute_branch_point_from_boxes(bounding_boxes):
    """
    Simple heuristic: take the intersection of the first two boxes
    (stem box and bar box) and use its center as the branch point.

    bounding_boxes: list of (xmin, xmax, ymin, ymax)
    """
    if bounding_boxes is None or len(bounding_boxes) < 2:
        raise ValueError("Need at least two bounding boxes to infer branch point.")

    (x1_min, x1_max, y1_min, y1_max), (x2_min, x2_max, y2_min, y2_max) = bounding_boxes[:2]

    ix_min = max(x1_min, x2_min)
    ix_max = min(x1_max, x2_max)
    iy_min = max(y1_min, y2_min)
    iy_max = min(y1_max, y2_max)

    if ix_min >= ix_max or iy_min >= iy_max:
        # fallback: midpoint of the two box centers
        cx1 = 0.5 * (x1_min + x1_max)
        cy1 = 0.5 * (y1_min + y1_max)
        cx2 = 0.5 * (x2_min + x2_max)
        cy2 = 0.5 * (y2_min + y2_max)
        return np.array([(cx1 + cx2) / 2.0, (cy1 + cy2) / 2.0], float)

    return np.array([(ix_min + ix_max) / 2.0, (iy_min + iy_max) / 2.0], float)


def build_tmaze_polylines_from_ports(
    port_rois,
    bounding_boxes=None,
    branch_point=None,
):
    """
    Construct TMaze polylines:

      BL: base -> branch -> left
      BR: base -> branch -> right
      LR: left -> branch -> right

    port_rois: dict like {'base': (xmin, xmax, ymin, ymax), 'left': (...), 'right': (...)}
    bounding_boxes: list of (xmin, xmax, ymin, ymax) (optional, for branch point)
    branch_point: optional (x, y) manual override.
    """
    # Extract centers
    for k in ["base", "left", "right"]:
        if k not in port_rois:
            raise ValueError(f"port_rois missing key {k!r}")
    base_xy = np.array(roi_center(port_rois["base"]), float)
    left_xy = np.array(roi_center(port_rois["left"]), float)
    right_xy = np.array(roi_center(port_rois["right"]), float)

    # Branch point
    if branch_point is None:
        if bounding_boxes is not None:
            branch_xy = _compute_branch_point_from_boxes(bounding_boxes)
        else:
            # crude fallback: average of the three port centers
            branch_xy = np.mean([base_xy, left_xy, right_xy], axis=0)
    else:
        branch_xy = np.array(branch_point[:2], float)

    poly_BL = np.vstack([base_xy, branch_xy, left_xy])
    poly_BR = np.vstack([base_xy, branch_xy, right_xy])
    poly_LR = np.vstack([left_xy, branch_xy, right_xy])

    return {
        "base": base_xy,
        "left": left_xy,
        "right": right_xy,
        "branch": branch_xy,
        "BL": poly_BL,
        "BR": poly_BR,
        "LR": poly_LR,
    }


In [ ]:
def linearize_and_bin_session(
    session_dir: str,
    dlc_pattern: str = "*_1100000_filtered.csv",
    output_dir: str | None = None,
    port_rois: dict[str, tuple[float, float, float, float]] | None = None,
    bounding_boxes: list[tuple[float, float, float, float]] | None = None,
    branch_point: tuple[float, float] | None = None,
    n_bins: int = 70, #70 bins comes to approx. 3 cm wide bins 
) -> str:
    """
    Use the cleaned, trimmed trial trajectories produced by process_session(...)
    to compute:

      - 1D linearized position along TMaze polylines
      - spatial bin index per frame

    Saves:
      YYYY-MM-DD_HH-MM-SS_TMaze_linearized_binned_trials.csv

    Returns:
      path to the saved CSV.
    """
    session_dir = os.path.abspath(session_dir)

    # --- figure out DLC + prefix (same logic as in process_session) ---
    dlc_dir = os.path.join(session_dir, "dlc")
    if output_dir is None:
        output_dir = dlc_dir
    os.makedirs(output_dir, exist_ok=True)

    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError(f"No DLC files matching {dlc_pattern!r} in {dlc_dir}")
    if len(dlc_files) > 1:
        print("Warning: multiple DLC files found, using the first one.")
    dlc_csv_path = dlc_files[0]

    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_TMaze"

    # --- load per-frame and per-trial data produced by process_session ---
    frames_path = os.path.join(output_dir, f"{prefix}_trimmed_positions_and_ports.csv")
    trials_clean_path = os.path.join(output_dir, f"{prefix}_trimmed_trial_locations_clean.csv")

    if not os.path.isfile(frames_path):
        raise FileNotFoundError(frames_path)
    if not os.path.isfile(trials_clean_path):
        raise FileNotFoundError(trials_clean_path)

    df_frames = pd.read_csv(frames_path)
    df_trials_clean = pd.read_csv(trials_clean_path)

    # --- port_rois & polylines ---
    if port_rois is None:
        port_rois = PORT_ROIS_DEFAULT.copy()

    polys = build_tmaze_polylines_from_ports(
        port_rois=port_rois,
        bounding_boxes=bounding_boxes,
        branch_point=branch_point,
    )

    # total lengths and bin edges per corridor
    lengths = {
        "BL": _polyline_total_length(polys["BL"]),
        "BR": _polyline_total_length(polys["BR"]),
        "LR": _polyline_total_length(polys["LR"]),
    }
    bin_edges = {
        corridor: np.linspace(0.0, L, n_bins + 1)
        for corridor, L in lengths.items()
    }

    # map trial_type -> corridor
    type_to_corridor = {
        "base_to_left": "BL",
        "left_to_base": "BL",
        "base_to_right": "BR",
        "right_to_base": "BR",
        "left_to_right": "LR",
        "right_to_left": "LR",
    }

    # --- linearize trial by trial ---
    all_rows = []
    for tid in sorted(df_trials_clean["trial_id"].unique()):
        sub = df_trials_clean[df_trials_clean["trial_id"] == tid].copy()
        if sub.empty:
            continue

        ttype = sub["trial_type"].iloc[0]
        corridor = type_to_corridor.get(ttype, None)

        XY = sub[["x_clean", "y_clean"]].to_numpy()

        if corridor is None:
            # Unknown / "other" trial types: fill NaNs / -1
            lin_pos = np.full(len(sub), np.nan, float)
            bin_idx = np.full(len(sub), -1, int)
        else:
            poly = polys[corridor]
            edges = bin_edges[corridor]

            lin_pos = linearize_positions_along_polyline(XY, poly)

            # assign bins; values outside [0, L] or with NaN -> -1
            bin_idx = np.digitize(lin_pos, edges) - 1  # 0..n_bins-1
            bad = (
                (bin_idx < 0)
                | (bin_idx >= n_bins)
                | np.isnan(lin_pos)
            )
            bin_idx = bin_idx.astype(int)
            bin_idx[bad] = -1

        sub["corridor"] = corridor
        sub["linear_pos"] = lin_pos
        sub["spatial_bin"] = bin_idx

        all_rows.append(sub)

    lin_df = pd.concat(all_rows, ignore_index=True)

    out_path = os.path.join(output_dir, f"{prefix}_linearized_binned_trials_{n_bins}nbins.csv")
    lin_df.to_csv(out_path, index=False)

    # optional: save bin edges for reference
    edges_meta = {
        "n_bins": n_bins,
        "lengths": lengths,
        "bin_edges_BL": bin_edges["BL"].tolist(),
        "bin_edges_BR": bin_edges["BR"].tolist(),
        "bin_edges_LR": bin_edges["LR"].tolist(),
    }
    meta_path = os.path.join(output_dir, f"{prefix}_linearized_bins_metadata_{n_bins}nbins.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(edges_meta, f, indent=2)

    print("Saved linearized + binned trials to:")
    print("  ", out_path)
    print("  ", meta_path)

    return out_path


In [ ]:
session_dir = "/media/toor/T7Shield/Mouse1637/Round1.5/TMaze/2024-06-14T17_19_31"
port_rois = PORT_ROIS_BY_ERA[era_for_session(session_dir)]  # this example session is aging2024

bounding_boxes = [
    (90, 1700, 60, 200),   # horizontal stem
    (830, 945, 80, 970),   # vertical bar
]

branch_point = (895, 110)  # optional manual override of branch point

# 1) make sure you've already run:
process_session(
    session_dir=session_dir,
    trim_info_name="dlc_trim_info.csv",
    dlc_pattern=os.path.join(session_dir, "dlc", "*_1100000_filtered.csv"),
    caiman_path=os.path.join(session_dir, "caiman", "caiman_results.hdf5"),
    port_rois=port_rois,
    bounding_boxes=bounding_boxes,
)

# 2) then linearize + bin:
linearize_and_bin_session(
    session_dir=session_dir,
    dlc_pattern="*_1100000_filtered.csv",
    port_rois=port_rois,
    bounding_boxes=bounding_boxes,
    n_bins=70,
    branch_point = (895, 110), 
)


## BATCH RUN

In [ ]:
from pathlib import Path
ROOT_DIR = Path("/media/toor/Seagate3/YOUNG_MICE/Mouse946/Round1.5/TMaze")  # <-- EDIT

# Exclude registration output folders (e.g. "<caiman_v>_Registration_Round<N>/"), which live
# alongside session-timestamp folders under Round<N>/<Task>/ per CLAUDE.md's "Session folder
# lifecycle". The previous filter
# ("caiman_final_*" not in name) was a literal substring check that could never match any real
# folder name, so it let such folders through silently; matched here against the same
# "Registration" substring the sibling Linear notebook already uses.
SESSION_DIRS = [Path(subdir) for subdir in ROOT_DIR.iterdir()
               if subdir.is_dir() and "Registration" not in subdir.name]

bounding_boxes = [
    (90, 1700, 60, 200),   # horizontal stem
    (830, 945, 80, 970),   # vertical bar
]


In [ ]:
for sess_dir in SESSION_DIRS:
    print(f"Processing session: {sess_dir}")
    port_rois = PORT_ROIS_BY_ERA[era_for_session(sess_dir)]
    process_session(
        session_dir=str(sess_dir),
        trim_info_name="dlc_trim_info.csv",
        dlc_pattern="*_filtered.csv",
        caiman_path=os.path.join(sess_dir, "caiman_final", "caiman_results.hdf5"),
        port_rois=port_rois,
        bounding_boxes=bounding_boxes,
    )
    linearize_and_bin_session(
        session_dir=str(sess_dir),
        dlc_pattern="*_1100000_filtered.csv",
        port_rois=port_rois,
        bounding_boxes=bounding_boxes,
        n_bins=70,
        branch_point = (895, 110), 
    )

## Sanity Check Plots

In [ ]:
import matplotlib.pyplot as plt

def plot_session_overview(
    loc_raw,
    loc_clean,
    p_trim,
    port_labels,
    port_rois,
    bounding_boxes=None,
    movement_mask=None,
    title=None,
):
    """
    Quick visual QC for a single session:
      - raw vs cleaned trajectory
      - port ROIs
      - bounding boxes
      - cleaned trajectory colored by port label, movement highlighted
    """
    loc_raw = np.asarray(loc_raw)
    loc_clean = np.asarray(loc_clean)
    p_trim = np.asarray(p_trim)
    port_labels = np.asarray(port_labels, dtype=object)
    if movement_mask is not None:
        movement_mask = np.asarray(movement_mask, dtype=bool)

    fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
    if title is not None:
        fig.suptitle(title)

    # -----------------------
    # Left: raw vs cleaned + ROIs / bounding boxes
    # -----------------------
    ax = axs[0]
    ax.set_title("Raw vs cleaned trajectory")

    # Subsample if insane number of points
    n = loc_raw.shape[0]
    max_points = 10000
    step = max(1, n // max_points)

    ax.plot(
        loc_raw[::step, 0],
        loc_raw[::step, 1],
        ".",
        alpha=0.2,
        markersize=2,
        label="raw",
    )
    ax.plot(
        loc_clean[::step, 0],
        loc_clean[::step, 1],
        ".",
        alpha=0.7,
        markersize=2,
        label="clean",
    )

    # Port ROIs as rectangles
    for name, (xmin, xmax, ymin, ymax) in port_rois.items():
        rect = plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, linestyle="--")
        ax.add_patch(rect)
        cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
        ax.text(cx, cy, name, fontsize=9, ha="center", va="center")

    # Bounding boxes as rectangles
    if bounding_boxes is not None:
        for (xmin, xmax, ymin, ymax) in bounding_boxes:
            w = xmax - xmin
            h = ymax - ymin
            rect = plt.Rectangle(
                (xmin, ymin),
                w,
                h,
                fill=False,
                linestyle=":",
            )
            ax.add_patch(rect)

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
    ax.legend(loc="best")

    # -----------------------
    # Right: cleaned trajectory colored by port label
    # -----------------------
    ax2 = axs[1]
    ax2.set_title("Cleaned trajectory (colored by port)")

    # Simple color map for ports
    port_colors = {
        None: "lightgrey",
        "base": "tab:blue",
        "left": "tab:green",
        "right": "tab:red",
    }

    # Plot per port label
    unique_ports = sorted(set(port_labels.tolist()), key=lambda x: str(x))
    for p in unique_ports:
        mask = port_labels == p
        if not np.any(mask):
            continue
        color = port_colors.get(p, "black")
        ax2.plot(
            loc_clean[mask, 0],
            loc_clean[mask, 1],
            ".",
            markersize=3,
            alpha=0.8,
            label=str(p),
            color=color,
        )

    # Optionally highlight movement frames (on top)
    if movement_mask is not None:
        mov = movement_mask.astype(bool)
        if np.any(mov):
            ax2.plot(
                loc_clean[mov, 0],
                loc_clean[mov, 1],
                ".",
                markersize=3,
                alpha=0.6,
                label="moving",
            )

    # Draw ROIs and boxes again for reference
    for name, (cx, cy, r) in port_rois.items():
        circ = plt.Circle((cx, cy), r, fill=False, linestyle="--")
        ax2.add_patch(circ)
        ax2.text(cx, cy, name, fontsize=9, ha="center", va="center")

    if bounding_boxes is not None:
        for (xmin, xmax, ymin, ymax) in bounding_boxes:
            w = xmax - xmin
            h = ymax - ymin
            rect = plt.Rectangle(
                (xmin, ymin),
                w,
                h,
                fill=False,
                linestyle=":",
            )
            ax2.add_patch(rect)

    ax2.set_xlabel("x")
    ax2.set_ylabel("y")
    ax2.set_aspect("equal")
    ax2.legend(loc="best")

    plt.tight_layout()
    plt.show()


def preview_session_geometry(
    session_dir: str,
    trim_info_name: str = "dlc_trim_info.csv",
    dlc_pattern: str = "*_1100000_filtered.csv",
    output_dir: str | None = None,
    port_rois: dict[str, tuple[float, float, float, float]] | None = None,
    bounding_boxes: list[tuple[float, float, float, float]] | None = None,
):
    """
    Load the *_trimmed_positions_and_ports.csv produced by process_session(...)
    and show overview plots.

    Use this AFTER you've run process_session(...) once for that session.
    """
    session_dir = os.path.abspath(session_dir)

    # 1) figure out dlc file (same logic as in process_session)
    dlc_dir = os.path.join(session_dir, "dlc")
    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError(f"No DLC files matching {dlc_pattern!r} in {dlc_dir}")
    if len(dlc_files) > 1:
        print("Warning: multiple DLC files found, using the first one.")
    dlc_csv_path = dlc_files[0]

    if output_dir is None:
        output_dir = dlc_dir

    # 2) reconstruct the frames CSV path
    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_TMaze"
    frames_path = os.path.join(output_dir, f"{prefix}_trimmed_positions_and_ports.csv")
    if not os.path.isfile(frames_path):
        raise FileNotFoundError(
            f"{frames_path} not found. Make sure you ran process_session(...) first."
        )

    # 3) load frame-wise data
    df = pd.read_csv(frames_path)

    loc_raw = df[["x_raw", "y_raw"]].to_numpy()
    loc_clean = df[["x_clean", "y_clean"]].to_numpy()
    p_trim = df["likelihood"].to_numpy()
    movement_mask = df["moving"].astype(bool).to_numpy()
    port_labels = df["port_label"].to_numpy()

    if port_rois is None:
        port_rois = PORT_ROIS_DEFAULT.copy()

    plot_session_overview(
        loc_raw=loc_raw,
        loc_clean=loc_clean,
        p_trim=p_trim,
        port_labels=port_labels,
        port_rois=port_rois,
        bounding_boxes=bounding_boxes,
        movement_mask=movement_mask,
        title=os.path.basename(session_dir),
    )


In [ ]:
preview_session_geometry(
    session_dir=session_dir,
    trim_info_name="dlc_trim_info.csv",
    dlc_pattern="*_1100000_filtered.csv",
    bounding_boxes=bounding_boxes,
    port_rois=port_rois,  # from PORT_ROIS_BY_ERA[era_for_session(session_dir)], set above
)


In [ ]:
def plot_trials_quick(
    session_dir,
    dlc_pattern="*_1100000_filtered.csv",
    output_dir=None,
    max_trials=12
):
    """
    Loads the *_trimmed_trial_locations_clean.csv file and plots
    the first N trials, each as a separate 2D line.
    Very lightweight, only for sanity checking.
    """
    import matplotlib.pyplot as plt

    # Resolve output directory
    dlc_dir = os.path.join(session_dir, "dlc")
    if output_dir is None:
        output_dir = dlc_dir

    # Resolve DLC file (for timestamp prefix)
    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError("No DLC files found.")
    dlc_csv_path = dlc_files[0]
    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_TMaze"

    # Load trial CSV
    clean_path = os.path.join(output_dir, f"{prefix}_trimmed_trial_locations_clean.csv")
    if not os.path.isfile(clean_path):
        raise FileNotFoundError(clean_path)

    df = pd.read_csv(clean_path)

    # Select first N trials
    trial_ids = sorted(df["trial_id"].unique())[:max_trials]

    # Plot
    fig, ax = plt.subplots(figsize=(6, 6))
    cmap = plt.get_cmap("tab10")

    for i, tid in enumerate(trial_ids):
        sub = df[df["trial_id"] == tid]
        ax.plot(
            sub["x_clean"],
            sub["y_clean"],
            "-",
            linewidth=2,
            alpha=0.8,
            color=cmap(i % 10),
            label=f"{tid}: {sub['trial_type'].iloc[0]}"
        )

    ax.set_title(f"First {len(trial_ids)} Trials (cleaned)")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
plot_trials_quick(
    "/media/toor/T7Shield/Mouse1637/Round1.5/TMaze/2024-06-14T17_19_31",
    max_trials=30   # or fewer for speed
)


In [ ]:
def plot_session_midpoint_check(
    session_dir: str,
    dlc_pattern: str = "*_1100000_filtered.csv",
    output_dir: str | None = None,
    port_rois: dict[str, tuple[float, float, float, float]] | None = None,
    bounding_boxes: list[tuple[float, float, float, float]] | None = None,
    max_points: int = 10000,
):
    """
    Mid-pipeline check:
      - cleaned trajectory
      - port ROIs
      - bounding boxes
      - moving vs non-moving (optional)
    Uses the *_trimmed_positions_and_ports.csv file produced by process_session.
    """
    session_dir = os.path.abspath(session_dir)
    dlc_dir = os.path.join(session_dir, "dlc")
    if output_dir is None:
        output_dir = dlc_dir

    # find DLC to get prefix
    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError(f"No DLC files matching {dlc_pattern!r} in {dlc_dir}")
    if len(dlc_files) > 1:
        print("Warning: multiple DLC files found, using the first one.")
    dlc_csv_path = dlc_files[0]

    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_TMaze"
    frames_path = os.path.join(output_dir, f"{prefix}_trimmed_positions_and_ports.csv")
    if not os.path.isfile(frames_path):
        raise FileNotFoundError(frames_path)

    df = pd.read_csv(frames_path)

    loc_clean = df[["x_clean", "y_clean"]].to_numpy()
    movement_mask = df["moving"].astype(bool).to_numpy()

    if port_rois is None:
        port_rois = PORT_ROIS_DEFAULT.copy()

    # subsample to avoid overplotting
    n = loc_clean.shape[0]
    step = max(1, n // max_points)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_title(f"Midpoint check: {os.path.basename(session_dir)}")

    # non-moving and moving in different styles
    idx_all = np.arange(n)
    idx_sub = idx_all[::step]

    not_moving = ~movement_mask
    nm_sub = idx_sub[not_moving[idx_sub]]
    mv_sub = idx_sub[movement_mask[idx_sub]]

    ax.plot(
        loc_clean[nm_sub, 0],
        loc_clean[nm_sub, 1],
        ".",
        alpha=0.2,
        markersize=2,
        label="not moving",
    )
    ax.plot(
        loc_clean[mv_sub, 0],
        loc_clean[mv_sub, 1],
        ".",
        alpha=0.7,
        markersize=2,
        label="moving",
    )

    # ports
    for name, (xmin, xmax, ymin, ymax) in port_rois.items():
        rect = plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, linestyle="--")
        ax.add_patch(rect)
        cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
        ax.text(cx, cy, name, fontsize=9, ha="center", va="center")

    # bounding boxes
    if bounding_boxes is not None:
        for (xmin, xmax, ymin, ymax) in bounding_boxes:
            w = xmax - xmin
            h = ymax - ymin
            rect = plt.Rectangle((xmin, ymin), w, h, fill=False, linestyle=":")
            ax.add_patch(rect)


    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()


In [ ]:
# 2. Midpoint 2D check
plot_session_midpoint_check(
    session_dir=session_dir,
    bounding_boxes=bounding_boxes,
)

In [ ]:
def plot_trials_quick(
    session_dir: str,
    dlc_pattern: str = "*_1100000_filtered.csv",
    output_dir: str | None = None,
    max_trials: int = 12,
):
    """
    Very quick trial-level 2D plot using the clean trimmed trial CSV.
    One line per trial, colored by trial id.
    """
    import matplotlib.pyplot as plt

    session_dir = os.path.abspath(session_dir)
    dlc_dir = os.path.join(session_dir, "dlc")
    if output_dir is None:
        output_dir = dlc_dir

    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError(f"No DLC files matching {dlc_pattern!r} in {dlc_dir}")
    dlc_csv_path = dlc_files[0]

    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_TMaze"
    clean_path = os.path.join(output_dir, f"{prefix}_trimmed_trial_locations_clean.csv")
    if not os.path.isfile(clean_path):
        raise FileNotFoundError(clean_path)

    df = pd.read_csv(clean_path)
    trial_ids = sorted(df["trial_id"].unique())[:max_trials]

    fig, ax = plt.subplots(figsize=(6, 6))
    cmap = plt.get_cmap("tab10")

    for i, tid in enumerate(trial_ids):
        sub = df[df["trial_id"] == tid]
        ax.plot(
            sub["x_clean"],
            sub["y_clean"],
            "-",
            linewidth=2,
            alpha=0.8,
            color=cmap(i % 10),
            label=f"{tid}: {sub['trial_type'].iloc[0]}",
        )

    ax.set_title(f"First {len(trial_ids)} trials (clean)")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_linearization_2d(
    session_dir: str,
    corridor: str = "BL",   # "BL", "BR", or "LR"
    dlc_pattern: str = "*_1100000_filtered.csv",
    output_dir: str | None = None,
    n_bins: int = 70,
):
    """
    Check linearization: show 2D cleaned positions colored by linear_pos
    for all frames in the chosen corridor.
    """
    import matplotlib.pyplot as plt

    session_dir = os.path.abspath(session_dir)
    dlc_dir = os.path.join(session_dir, "dlc")
    if output_dir is None:
        output_dir = dlc_dir

    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError(f"No DLC files matching {dlc_pattern!r} in {dlc_dir}")
    dlc_csv_path = dlc_files[0]

    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_TMaze"
    lin_path = os.path.join(output_dir, f"{prefix}_linearized_binned_trials_{n_bins}nbins.csv")
    meta_path = os.path.join(output_dir, f"{prefix}_linearized_bins_metadata_{n_bins}nbins.json")

    if not os.path.isfile(lin_path):
        raise FileNotFoundError(lin_path)
    if not os.path.isfile(meta_path):
        raise FileNotFoundError(meta_path)

    df = pd.read_csv(lin_path)
    df_corr = df[df["corridor"] == corridor].copy()
    if df_corr.empty:
        raise ValueError(f"No data found for corridor {corridor!r}")

    # 2D scatter colored by linear_pos
    fig, ax = plt.subplots(figsize=(6, 6))
    sc = ax.scatter(
        df_corr["x_clean"],
        df_corr["y_clean"],
        c=df_corr["linear_pos"],
        s=4,
        alpha=0.7,
    )
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Linearization check: corridor {corridor}")
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("linear position")
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_linear_pos_per_trial(
    session_dir: str,
    trial_types: list[str] | None = None,
    n_trials_per_type: int = 3,
    dlc_pattern: str = "*_1100000_filtered.csv",
    output_dir: str | None = None,
    n_bins: int = 70,
):
    """
    For a few example trials, show linear_pos vs frame_in_trial and
    the corresponding spatial_bin. Good to verify binning looks sane.
    """
    import matplotlib.pyplot as plt

    session_dir = os.path.abspath(session_dir)
    dlc_dir = os.path.join(session_dir, "dlc")
    if output_dir is None:
        output_dir = dlc_dir

    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError(f"No DLC files matching {dlc_pattern!r} in {dlc_dir}")
    dlc_csv_path = dlc_files[0]

    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_TMaze"
    lin_path = os.path.join(output_dir, f"{prefix}_linearized_binned_trials_{n_bins}nbins.csv")
    if not os.path.isfile(lin_path):
        raise FileNotFoundError(lin_path)

    df = pd.read_csv(lin_path)

    # Which trial types to show
    if trial_types is None:
        trial_types = sorted(df["trial_type"].dropna().unique().tolist())

    fig, axes = plt.subplots(len(trial_types), 1, figsize=(8, 3 * len(trial_types)), sharex=False)
    if len(trial_types) == 1:
        axes = [axes]

    for ax, ttype in zip(axes, trial_types):
        df_type = df[df["trial_type"] == ttype]
        trial_ids = sorted(df_type["trial_id"].unique())[:n_trials_per_type]

        for tid in trial_ids:
            sub = df_type[df_type["trial_id"] == tid]
            ax.plot(
                sub["frame_in_trial"],
                sub["linear_pos"],
                "-",
                alpha=0.8,
                label=f"trial {tid}",
            )

        ax.set_ylabel("linear_pos")
        ax.set_title(f"trial_type = {ttype}")
        ax.legend(loc="best")

    axes[-1].set_xlabel("frame_in_trial")
    plt.tight_layout()
    plt.show()

    # Optional: same for bin index
    fig2, axes2 = plt.subplots(len(trial_types), 1, figsize=(8, 3 * len(trial_types)), sharex=False)
    if len(trial_types) == 1:
        axes2 = [axes2]

    for ax, ttype in zip(axes2, trial_types):
        df_type = df[df["trial_type"] == ttype]
        trial_ids = sorted(df_type["trial_id"].unique())[:n_trials_per_type]

        for tid in trial_ids:
            sub = df_type[df_type["trial_id"] == tid]
            ax.step(
                sub["frame_in_trial"],
                sub["spatial_bin"],
                where="mid",
                alpha=0.8,
                label=f"trial {tid}",
            )

        ax.set_ylabel("bin")
        ax.set_title(f"bins, trial_type = {ttype}")
        ax.legend(loc="best")

    axes2[-1].set_xlabel("frame_in_trial")
    plt.tight_layout()
    plt.show()


In [ ]:
session_dir = "/media/toor/T7Shield/Mouse1637/Round1.5/TMaze/2024-06-14T17_19_31"
port_rois = PORT_ROIS_BY_ERA[era_for_session(session_dir)]  # this example session is aging2024

bounding_boxes = [
    (90, 1700, 60, 200),   # horizontal stem
    (830, 945, 80, 970),   # vertical bar
]

# 1. Run the main pipeline once (if not already done)
process_session(
    session_dir=session_dir,
    trim_info_name="dlc_trim_info.csv",
    dlc_pattern="*_1100000_filtered.csv",
    caiman_path=os.path.join(session_dir, "caiman", "caiman_results.hdf5"),
    port_rois=port_rois,
    bounding_boxes=bounding_boxes,
)

# 2. Midpoint 2D check
plot_session_midpoint_check(
    session_dir=session_dir,
    port_rois=port_rois,
    bounding_boxes=bounding_boxes,
)

# 3. Trial-wise 2D check
plot_trials_quick(session_dir=session_dir, max_trials=10)

# 4. Linearize + bin (end result)
linearize_and_bin_session(
    session_dir=session_dir,
    port_rois=port_rois,
    bounding_boxes=bounding_boxes,
    n_bins=70,
)

# 5. End-result checks
plot_linearization_2d(session_dir=session_dir, corridor="BR")  # or "BR", "LR"
plot_linear_pos_per_trial(session_dir=session_dir, n_trials_per_type=3)


In [ ]:
# 2. Midpoint 2D check
plot_session_midpoint_check(
    session_dir=session_dir,
    port_rois=port_rois,
    bounding_boxes=bounding_boxes)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_tmaze_branch_and_polylines(
    session_dir: str,
    dlc_pattern: str = "*_1100000_filtered.csv",
    output_dir: str | None = None,
    port_rois: dict[str, tuple[float, float, float, float]] | None = None,
    bounding_boxes: list[tuple[float, float, float, float]] | None = None,
    branch_point: tuple[float, float] | None = None,
    sample_frac: float = 1.0,
):
    """
    Visualize TMaze geometry for a session:
      - cleaned positions (from *_trimmed_positions_and_ports.csv)
      - port centers
      - branch point with crosshairs and coordinates
      - BL, BR, LR polylines

    sample_frac: fraction of frames to plot (1.0 = all, 0.1 = 10 percent)
    """

    session_dir = os.path.abspath(session_dir)
    dlc_dir = os.path.join(session_dir, "dlc")
    if output_dir is None:
        output_dir = dlc_dir

    # Find DLC file to reconstruct prefix
    dlc_files = glob.glob(os.path.join(dlc_dir, dlc_pattern))
    if not dlc_files:
        raise FileNotFoundError(f"No DLC files matching {dlc_pattern!r} in {dlc_dir}")
    if len(dlc_files) > 1:
        print("Warning: multiple DLC files found, using the first one.")
    dlc_csv_path = dlc_files[0]

    stamp = extract_datetime_stamp(dlc_csv_path)
    prefix = f"{stamp}_TMaze"

    frames_path = os.path.join(output_dir, f"{prefix}_trimmed_positions_and_ports.csv")
    if not os.path.isfile(frames_path):
        raise FileNotFoundError(frames_path)

    df_frames = pd.read_csv(frames_path)

    # Port ROIs and polylines
    if port_rois is None:
        port_rois = PORT_ROIS_DEFAULT.copy()

    polys = build_tmaze_polylines_from_ports(
        port_rois=port_rois,
        bounding_boxes=bounding_boxes,
        branch_point=branch_point,
    )

    base_xy   = polys["base"]
    left_xy   = polys["left"]
    right_xy  = polys["right"]
    branch_xy = polys["branch"]

    # Optionally subsample frames to keep plot light
    if 0 < sample_frac < 1.0:
        df_plot = df_frames.sample(frac=sample_frac, random_state=0)
    else:
        df_plot = df_frames

    x_clean = df_plot["x_clean"].to_numpy()
    y_clean = df_plot["y_clean"].to_numpy()

    fig, ax = plt.subplots(figsize=(6, 6))

    # Background: cleaned trajectory
    ax.scatter(x_clean, y_clean, s=2, alpha=0.3, label="cleaned trajectory")

    # Plot polylines
    for key, color, label in [("BL", "tab:blue", "BL base-branch-left"),
                              ("BR", "tab:orange", "BR base-branch-right"),
                              ("LR", "tab:green", "LR left-branch-right")]:
        poly = polys[key]
        ax.plot(poly[:, 0], poly[:, 1], "-o", label=label)

    # Plot ports
    ax.scatter(base_xy[0], base_xy[1], marker="s", s=80, color="k")
    ax.text(base_xy[0], base_xy[1], f" base\n({base_xy[0]:.1f}, {base_xy[1]:.1f})",
            ha="left", va="bottom", fontsize=8)

    ax.scatter(left_xy[0], left_xy[1], marker="s", s=80, color="k")
    ax.text(left_xy[0], left_xy[1], f" left\n({left_xy[0]:.1f}, {left_xy[1]:.1f})",
            ha="left", va="bottom", fontsize=8)

    ax.scatter(right_xy[0], right_xy[1], marker="s", s=80, color="k")
    ax.text(right_xy[0], right_xy[1], f" right\n({right_xy[0]:.1f}, {right_xy[1]:.1f})",
            ha="left", va="bottom", fontsize=8)

    # Branch point with crosshairs
    ax.scatter(branch_xy[0], branch_xy[1], marker="x", s=80, color="red", zorder=5)
    ax.axvline(branch_xy[0], color="red", linestyle="--", alpha=0.6)
    ax.axhline(branch_xy[1], color="red", linestyle="--", alpha=0.6)
    ax.text(branch_xy[0], branch_xy[1],
            f" branch\n({branch_xy[0]:.1f}, {branch_xy[1]:.1f})",
            ha="left", va="bottom", color="red", fontsize=8)

    # Optional: draw bounding boxes
    if bounding_boxes is not None:
        for (xmin, xmax, ymin, ymax) in bounding_boxes:
            ax.plot([xmin, xmax, xmax, xmin, xmin],
                    [ymin, ymin, ymax, ymax, ymin],
                    linestyle=":", color="gray", alpha=0.7)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
    ax.set_title("TMaze geometry: ports, branch point, polylines")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()
